# **SETUP**

In [4]:
import pathlib
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

categories = list(train.select_dtypes("category").columns)
cbc = CatBoostClassifier(
    thread_count=-1,
    cat_features=categories,
    verbose=0,
    allow_writing_files=False,
    random_seed=123
)

# **OOF PREDICTIONS**

In [5]:
scores = []
oof = pd.Series(index=train.index, dtype=float, name="oof")

X = train.drop(columns=["PitNextLap"])
y = train["PitNextLap"]

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    cbc.fit(X[~is_val], y[~is_val])

    oof[is_val] = cbc.predict_proba(X[is_val])[:, 1]
    scores.append(roc_auc_score(y[is_val], oof[is_val]))

print(scores)
print("OOF ROC-AUC:", roc_auc_score(y, oof))

[0.9498452740864902, 0.9487468442684933, 0.9474364612020183, 0.9467778473817591, 0.9484603622634356]
OOF ROC-AUC: 0.9482485917977727


# **TEST PREDICTIONS**

In [6]:
cbc.fit(X, y)
preds = pd.Series(cbc.predict_proba(test)[:, 1], index=test.index, dtype=float, name="preds")

# **EXPORT**

In [7]:
model_name = "000-cbc-baseline"
(PROJECT_ROOT / "data" / "predictions" / model_name).mkdir(exist_ok=True)

oof.to_frame().to_parquet(PROJECT_ROOT / "data" / "predictions" / model_name / "train.parquet")
preds.to_frame().to_parquet(PROJECT_ROOT / "data" / "predictions" / model_name / "test.parquet")